In [1]:
import torch
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

X = housing['data']
y = housing['target']

X_train_full, X_test, y_train_full, y_test = train_test_split(X,y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full,y_train_full)

print(X_train.shape, X_test.shape, X_valid.shape)

scl = StandardScaler()
scl.fit(X_train)

X_train = scl.transform(X_train)
X_test = scl.transform(X_test)
X_valid = scl.transform(X_valid)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
X_valid = torch.FloatTensor(X_valid)

y_train = torch.FloatTensor(y_train).view(-1,1)
y_test = torch.FloatTensor(y_test).view(-1,1)
y_valid = torch.FloatTensor(y_valid).view(-1,1)

(11610, 8) (5160, 8) (3870, 8)


In [14]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
valid_dataset = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)
valid_loader = DataLoader(valid_dataset, batch_size=32)

In [17]:
import torch.nn as nn
import torchmetrics
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cpu'

In [20]:
len(train_loader)

363

In [27]:
learning_rate = 0.01
model = nn.Sequential(
	nn.Linear(in_features=8, out_features=30), 
	nn.Sigmoid(),
	nn.Linear(in_features=30, out_features=50), 
	nn.Sigmoid(),
	nn.Linear(in_features=50, out_features=1)
)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)
metric = torchmetrics.MeanAbsoluteError()

history = {
	'loss' : [],
	'train_metric' : [],
	'valid_metric' : []
}
n_epochs = 10

for epoch in range(n_epochs):
	# Training 
	total_loss = 0
	metric.reset()
	model.train()
	for X_batch, y_batch in train_loader:
		y_pred = model(X_batch)
		loss = criterion(y_pred, y_batch)
		total_loss += loss.item()
		loss.backward()
		optimizer.step()
		optimizer.zero_grad()
		metric.update(y_pred, y_batch)

	avg_loss = total_loss / len(train_loader)
	history['loss'].append(avg_loss)

	avg_metric_train = metric.compute().item()
	history['train_metric'].append(avg_metric_train)

	# Evaluation
	model.eval()
	metric.reset()

	with torch.no_grad():
		for X_batch, y_batch in valid_loader:
			y_pred = model(X_batch)
			metric.update(y_pred, y_batch)

	avg_metric_valid = metric.compute().item()
	history['valid_metric'].append(avg_metric_valid)

	print(
		f'Epoch: {epoch+1}/{n_epochs}, '
		+f'Loss: {round(avg_loss,3)}, '
		+f'Train Metric: {round(avg_metric_train,3)}, ' 
		+f'Valid Metric: {round(avg_metric_valid,3)}'
	)

Epoch: 1/10, Loss: 1.329, Train Metric: 0.908, Valid Metric: 0.906
Epoch: 2/10, Loss: 1.25, Train Metric: 0.885, Valid Metric: 0.902
Epoch: 3/10, Loss: 1.134, Train Metric: 0.841, Valid Metric: 0.809
Epoch: 4/10, Loss: 0.91, Train Metric: 0.748, Valid Metric: 0.737
Epoch: 5/10, Loss: 0.714, Train Metric: 0.652, Valid Metric: 0.615
Epoch: 6/10, Loss: 0.644, Train Metric: 0.608, Valid Metric: 0.597
Epoch: 7/10, Loss: 0.609, Train Metric: 0.585, Valid Metric: 0.586
Epoch: 8/10, Loss: 0.585, Train Metric: 0.57, Valid Metric: 0.549
Epoch: 9/10, Loss: 0.563, Train Metric: 0.557, Valid Metric: 0.535
Epoch: 10/10, Loss: 0.545, Train Metric: 0.546, Valid Metric: 0.538
